# 作业 03：在 PyTorch 中调用交叉熵

本次只判断一件事：**有没有行人？** 不使用图片、CNN 或数据下载。我们用一个简单函数产生两个预测概率，把注意力放在最后的损失函数调用上。

类别的顺序始终固定：

| 类别编号 | 含义 | one-hot 标签 y |
|---|---|---|
| 0 | 没有行人 | `[1, 0]` |
| 1 | 有行人 | `[0, 1]` |

`y` 和 `y_pred` 都表示**同一个样本的两个类别**，shape 为 `(2,)`。`y_pred[0]` 是没有行人的概率，`y_pred[1]` 是有行人的概率，两者相加为 1。

目标：会调用 `F.cross_entropy`，并能解释概率变大或变小时，损失为什么变化。


In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

def show(name, value):
    print(f'{name}: {value}, shape={tuple(value.shape)}, dtype={value.dtype}')


## 1. 一个输出两个概率的预测函数

为了便于观察，直接把输入 x 当作“有行人的预测概率”，于是“没有行人的概率”就是 `1-x`。这是人为设置的教学函数，不是训练好的行人识别器。

先取 x=0.2，则预测应为 `[0.8, 0.2]`。


In [2]:
def predict(x):
    # x 是一个 0 到 1 之间的浮点标量张量
    return torch.stack([1 - x, x])

x = torch.tensor(0.2)
y = torch.tensor([1., 0.])  # 真实情况：没有行人
y_pred = predict(x)

show('x', x)
show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)
show('预测概率之和', y_pred.sum())


x: 0.20000000298023224, shape=(), dtype=torch.float32
y（one-hot 真实标签）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32


## 2. 最后调用 cross_entropy 计算损失

**PyTorch 的 `F.cross_entropy` 接收原始分数 logits，而不是直接接收概率。**

本例的函数已经输出概率，因此先用 `torch.log(y_pred)` 构造一组等价分数：对这组分数做 softmax，会还原原来的概率。这里两个概率均大于 0。

真实标签使用类别编号：`y=[1,0]` 对应编号 `0`，`y=[0,1]` 对应编号 `1`。用 `y.argmax()` 取出这个编号，得到整数标量。这里取的是**真实标签**的编号，不是预测的 argmax。

请重点看最后一行：损失由 `F.cross_entropy(...)` 计算。


In [3]:
scores = torch.log(y_pred)  # 从本例的概率构造等价 logits
label = y.argmax()        # [1, 0] → 类别编号 0，dtype 为 torch.int64

show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))
show('label（真实类别编号）', label)

loss = F.cross_entropy(scores, label)
show('loss', loss)


scores（传给损失函数的分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss: 0.2231435477733612, shape=(), dtype=torch.float32


### 对照手算

展开公式：

$$L=-[y_0\ln(p_0)+y_1\ln(p_1)].$$

真实标签为 `[1,0]` 时，只留下 $-\ln(p_0)$。当前 $p_0=0.8$，所以损失约为 **0.2231**。

下面仅用于核对；实际调用损失函数时，仍使用上面的 `F.cross_entropy`。


In [4]:
manual_loss = -torch.log(y_pred[label])
show('手算：真实类别概率的负对数', manual_loss)
print('与 cross_entropy 一致：', torch.allclose(loss, manual_loss))


手算：真实类别概率的负对数: 0.2231435328722, shape=(), dtype=torch.float32
与 cross_entropy 一致： True


### 实验 1：把第 1 节的 x 改成 0.8

真实标签仍为 `[1,0]`（“没有行人”），只把预测函数的输入 `x` 从 `0.2` 改成 `0.8`，然后从第 1 节重跑一遍：预测 → 由概率构造等价分数 → 取真实类别编号 → 调用 `cross_entropy` → 与手算 `-ln(p_true)` 核对。

预期：`y_pred = [0.2, 0.8]`，真实类别概率 `p0 = 0.2`，损失 `-ln(0.2) ≈ 1.6094`，比 `x = 0.2` 时的 `0.2231` 大约 7.2 倍。


In [5]:
# 实验 1：把第 1 节的 x 改成 0.8，其余步骤与第 1、2 节完全相同
x_exp = torch.tensor(0.8)          # 第 1 节中被改为 0.8 的那个 x
y_exp = torch.tensor([1., 0.])     # 真实情况仍然是没有行人
y_pred_exp = predict(x_exp)        # [1 - x, x]

show('x（改为 0.8）', x_exp)
show('y（one-hot 真实标签）', y_exp)
show('y_pred（两个预测概率）', y_pred_exp)
show('预测概率之和', y_pred_exp.sum())

scores_exp = torch.log(y_pred_exp)   # 概率 → 等价 logits，softmax 后还原概率
label_exp = y_exp.argmax()           # [1, 0] → 类别编号 0
show('scores（传给损失函数的分数）', scores_exp)
show('softmax(scores)（还原概率）', scores_exp.softmax(dim=0))
show('label（真实类别编号）', label_exp)

loss_exp = F.cross_entropy(scores_exp, label_exp)
show('loss（x = 0.8 时的交叉熵）', loss_exp)

manual_loss_exp = -torch.log(y_pred_exp[label_exp])   # 手算 -ln(p0)
show('手算：真实类别概率的负对数', manual_loss_exp)
print('与 cross_entropy 一致：', torch.allclose(loss_exp, manual_loss_exp))

baseline = -torch.log(predict(torch.tensor(0.2))[0])  # x = 0.2 时真实类别（0）的损失 = -ln(0.8)
print(f'x = 0.2 时 loss = {baseline.item():.4f}，'
      f'x = 0.8 时 loss = {loss_exp.item():.4f}，'
      f'后者是前者的 {loss_exp.item() / baseline.item():.2f} 倍')


x（改为 0.8）: 0.800000011920929, shape=(), dtype=torch.float32
y（one-hot 真实标签）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.2000, 0.8000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32
scores（传给损失函数的分数）: tensor([-1.6094, -0.2231]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.2000, 0.8000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss（x = 0.8 时的交叉熵）: 1.6094379425048828, shape=(), dtype=torch.float32
手算：真实类别概率的负对数: 1.6094379425048828, shape=(), dtype=torch.float32
与 cross_entropy 一致： True
x = 0.2 时 loss = 0.2231，x = 0.8 时 loss = 1.6094，后者是前者的 7.21 倍


## 3. 几个带数字的例子

下面固定真实标签为 `[1,0]`，即“没有行人”。只改变模型的预测概率。

| 真实标签 y | 预测概率 y_pred | 交叉熵 |
|---|---|---|
| `[1,0]` | `[0.9,0.1]` | $-\ln(0.9)\approx0.1054$ |
| `[1,0]` | `[0.5,0.5]` | $-\ln(0.5)\approx0.6931$ |
| `[1,0]` | `[0.1,0.9]` | $-\ln(0.1)\approx2.3026$ |
| `[1,0]` | `[0,1]` | $-\ln(0)\to+\infty$：把真实类别的概率设为 0 |
| `[1,0]` | `[1,0]` | $-\ln(1)=0$：完全相信真实类别 |

概率越接近真实标签，损失越小。概率精确为 0 的情况是边界示例；PyTorch 会在下面的错误预测例子中打印 `inf`，表示正无穷。


In [6]:
def compare(y, probabilities):
    label = y.argmax()
    # 仅为展示指定概率的交叉熵。log(0)=-inf，是这里刻意保留的边界情况。
    scores = torch.log(probabilities)
    loss = F.cross_entropy(scores, label)
    print(f'y={y.tolist()}, y_pred={probabilities.tolist()}')
    print(f'真实类别概率={probabilities[label].item():.4f}, loss={loss.item():.4f}')
    return loss

y = torch.tensor([1., 0.])
example_losses = []
for values in [[0.9, 0.1], [0.5, 0.5], [0.1, 0.9], [0., 1.], [1., 0.]]:
    example_losses.append(compare(y, torch.tensor(values)))
    print()


y=[1.0, 0.0], y_pred=[0.8999999761581421, 0.10000000149011612]
真实类别概率=0.9000, loss=0.1054

y=[1.0, 0.0], y_pred=[0.5, 0.5]
真实类别概率=0.5000, loss=0.6931

y=[1.0, 0.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.1000, loss=2.3026

y=[1.0, 0.0], y_pred=[0.0, 1.0]
真实类别概率=0.0000, loss=inf

y=[1.0, 0.0], y_pred=[1.0, 0.0]
真实类别概率=1.0000, loss=0.0000



## 4. 换成“有行人”试试看

现在 y 为 `[0,1]`，真实类别是 1，应当关注第二个概率。下面预测为 `[0.1,0.9]`，损失约为 **0.1054**。同一个预测在上节被惩罚，在这里得到较小损失，是因为真实答案变了。


In [7]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.1, 0.9])
compare(y, y_pred)


y=[0.0, 1.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.9000, loss=0.1054


tensor(0.1054)

## 5. 真正的网络输出原始分数时，直接调用

前面的 `torch.log` 是为了配合本题“给定概率”的数值演示。真实网络通常直接输出两个原始分数，此时**直接把分数交给 cross_entropy**，不需要先 softmax 或取 log。

下面的 scores 为 `[2,-1]`，它们不是概率，也不需要加起来等于 1。softmax 只用来把概率打印给我们看，不是交叉熵的输入。


In [8]:
scores = torch.tensor([2., -1.])
y = torch.tensor([1., 0.])
label = y.argmax()

show('原始分数 scores', scores)
show('给人看的预测概率', scores.softmax(dim=0))
loss = F.cross_entropy(scores, label)
show('直接调用 cross_entropy 得到的 loss', loss)


原始分数 scores: tensor([ 2., -1.]), shape=(2,), dtype=torch.float32
给人看的预测概率: tensor([0.9526, 0.0474]), shape=(2,), dtype=torch.float32
直接调用 cross_entropy 得到的 loss: 0.04858732968568802, shape=(), dtype=torch.float32


## 你的实验（50分）

1. 把第 1 节的 x 改成 `0.8`，从第 1 节重新运行到手算核对。保持真实标签为 `[1,0]`。记录 y_pred 和损失，解释为什么损失变大。
2. 在下面补全代码：真实标签是“有行人”，预测概率为 `[0.8,0.2]`。打印两个张量的 shape，再调用 cross_entropy，预期损失约为 `1.6094`。
3. 用自己的话解释：为什么 `[1,0]` 对 `[0,1]` 的损失是无穷大，而 `[1,0]` 对 `[1,0]` 的损失是 0？
4. `F.cross_entropy(y_pred, label)` 与 `F.cross_entropy(torch.log(y_pred), label)` 中，哪一个才是在计算这里指定的预测概率对应的交叉熵？原始网络输出 scores 时，又该怎样调用？


In [9]:
y = torch.tensor([0., 1.])            # 真实标签：「有行人」→ 类别编号 1
y_pred = torch.tensor([0.8, 0.2])     # 预测概率：没有行人 0.8，有行人 0.2

print('y shape:', tuple(y.shape), ' y_pred shape:', tuple(y_pred.shape))
print('y =', y, ' y_pred =', y_pred)

label = y.argmax()                    # 由 one-hot 真实标签得到类别编号，dtype 为 torch.int64
show('label（真实类别编号）', label)

scores = torch.log(y_pred)            # 概率 → 等价 logits
show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))

loss = F.cross_entropy(scores, label)
show('loss', loss)

print('真实类别概率 p1 =', y_pred[label].item(),
      ' 手算 -ln(p1) =', (-torch.log(y_pred[label])).item())


y shape: (2,)  y_pred shape: (2,)
y = tensor([0., 1.])  y_pred = tensor([0.8000, 0.2000])
label（真实类别编号）: 1, shape=(), dtype=torch.int64
scores（传给损失函数的分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
loss: 1.6094379425048828, shape=(), dtype=torch.float32
真实类别概率 p1 = 0.20000000298023224  手算 -ln(p1) = 1.6094379425048828


### 我的回答 (50分)

**1. 修改 x 后的预测与损失（实验 1）**

把第 1 节的 `x` 从 `0.2` 改成 `0.8`，真实标签保持 `[1,0]`，重跑第 1、2 节得到：`y_pred = predict(0.8) = [0.2, 0.8]`（shape `(2,)`）、`scores = log(y_pred) = [-1.6094, -0.2231]`、`label = 0`、`loss = F.cross_entropy(scores, 0) = 1.6094`，与手算 `-ln(p0) = -ln(0.2)` 一致（`allclose` 为 True）。

**损失变大是因为交叉熵只惩罚“真实类别那一项概率”**：真实是“没有行人”，正确类别是 0，而模型给类别 0 的概率从 `0.8` 掉到 `0.2`，`-ln(p0)` 就从 `0.2231` 升到 `1.6094`，约 7.21 倍。直观地说，`x` 表示“有行人的概率”，`x = 0.8` 等于模型在这个“没有行人”的样本上答错了方向，所以惩罚更重。反过来，若真实标签换成 `[0,1]`（有行人），同一个 `y_pred = [0.2, 0.8]` 的损失只有 `-ln(0.8) ≈ 0.2231`——同一个预测，真实答案变了，损失也跟着变，这正是“交叉熵在衡量预测分布与真实标签的差距”。

**2. “有行人”实验的结果**

`y = [0,1]`、`y_pred = [0.8, 0.2]`，两者 shape 都是 `(2,)`：这两个数属于**同一个样本的两个类别**，不是两个样本，且概率和为 1。由 one-hot 标签取编号 `label = y.argmax() = 1`（`torch.int64` 标量）。把概率转成等价分数 `scores = log(y_pred) = [-0.2231, -1.6094]` 后调用 `F.cross_entropy(scores, 1)`，得到 `loss = 1.6094`，与预期一致，也等于手算 `-ln(p1) = -ln(0.2)`。这里被惩罚的是第二个概率：模型认为“有行人”只有 0.2，而真实答案恰恰是“有行人”，所以损失较大。

**3. 两个极端例子的解释**

交叉熵的公式是 `L = -[y0·ln(p0) + y1·ln(p1)]`，one-hot 标签会让其中一项消失，只留下真实类别的 `-ln(p_true)`，所以损失只取决于**正确类别的概率**。

- `[1,0]` 对 `[0,1]`：真实类别是 0，模型却给出 `p0 = 0`，`-ln(0) → +∞`。这是“把正确答案的概率压成 0”的极端错误，惩罚无穷大，因此显示 `inf`（这是刻意演示的边界情况，不是代码错误；也说明训练中不能让概率真变成 0，log 会溢出）。
- `[1,0]` 对 `[1,0]`：`p0 = 1`，`-ln(1) = 0`。模型把全部概率押在正确类别上，没有任何不确定性，损失取下界 0。

中间地带的规律由上面的例子给出：`p_true` 为 0.9 → 0.1054、0.5 → 0.6931、0.1 → 2.3026，概率越接近 1 损失越接近 0，越接近 0 损失增长越快。所以“错得很自信”（`inf`）的代价远大于“模棱两可”（0.6931），这也是交叉熵鼓励模型给正确类别高分的原因。

**4. `F.cross_entropy(y_pred, label)` 还是 `F.cross_entropy(torch.log(y_pred), label)`**

`F.cross_entropy` 的第一个参数要的是**原始分数 logits**，它在内部自己做 `log_softmax`，不会把输入当成概率。

- 本题给定概率时，正确的是 **`F.cross_entropy(torch.log(y_pred), label)`**：因为 `softmax(log p) = p`，这组分数恰好还原原来的预测概率，结果就等于 `-ln(p_true)`（`[0.8,0.2]` → `0.2231`，`[0.2,0.8]` → `1.6094`）。
- `F.cross_entropy(y_pred, label)` 是**误用**：它会把 `0.8`、`0.2` 当成 logits 再 softmax 一次，得到 `[0.6457, 0.3543]` 这组新概率，损失变成 `0.4375`，已经不是本题预测概率对应的交叉熵了。
- 真实网络直接输出两个分数时（第 5 节），**直接把 scores 交给 `F.cross_entropy(scores, label)`**，不要再手动 softmax 或取 log；`softmax` 只用来把概率打印出来给人看。`label` 必须是由 one-hot 标签经 `argmax()` 得到的整数类别编号，而不是 one-hot 向量。
